# E3 ablation evaluation

This notebook compares E0, E2 filter-only, E0 abstain-only, and full E2 using only cached predictions and cached Qwen passage scores from E2's results folder. No generation and no GPU are needed.

`rgb_e2_e2_predictions.jsonl` is saved before abstention (filter + generate only). Abstention is applied afterwards by replacing the answer with the refusal sentence when the best original-passage score is strictly less than `tau`, where `tau` is `abstention_threshold` from `rgb_e2_qwen_metrics.json`.


In [1]:
from pathlib import Path
import hashlib, json
from collections import Counter, defaultdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "E2_Qwen_F1").exists():
    REPO_ROOT = REPO_ROOT.parent
E2_RESULTS = REPO_ROOT / "E2_Qwen_F1" / "results_e2_qwen_f1"
E3_RESULTS = REPO_ROOT / "results_e3_ablation"
E3_RESULTS.mkdir(exist_ok=True)
def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def read_json(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))
paths = {name: E2_RESULTS / name for name in (
    "shared_contexts.jsonl", "shared_contexts_manifest.json",
    "rgb_e2_passage_scores.jsonl", "rgb_e2_e0_predictions.jsonl",
    "rgb_e2_e2_predictions.jsonl", "rgb_e2_e0_predictions.jsonl.meta.json",
    "rgb_e2_e2_predictions.jsonl.meta.json")}
contexts = read_jsonl(paths["shared_contexts.jsonl"])
manifest = read_json(paths["shared_contexts_manifest.json"])
scores = read_jsonl(paths["rgb_e2_passage_scores.jsonl"])
e0 = read_jsonl(paths["rgb_e2_e0_predictions.jsonl"])
e2 = read_jsonl(paths["rgb_e2_e2_predictions.jsonl"])
e0_meta = read_json(paths["rgb_e2_e0_predictions.jsonl.meta.json"])
e2_meta = read_json(paths["rgb_e2_e2_predictions.jsonl.meta.json"])
print({"contexts": len(contexts), "scores": len(scores), "e0": len(e0), "e2_cache": len(e2)})

{'contexts': 1300, 'scores': 6489, 'e0': 1300, 'e2_cache': 1300}


In [2]:
def row_key(row):
    return (row["condition"], str(row["id"]))

def context_hash(rows):
    text = "".join(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n" for row in rows)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

expected = {"noise_0.0": 300, "noise_0.4": 300, "noise_0.8": 300, "incomplete": 300, "contradictory": 100}
test_expected = {"noise_0.0": 210, "noise_0.4": 210, "noise_0.8": 210, "incomplete": 210, "contradictory": 70}
assert Counter(row["condition"] for row in contexts) == Counter(expected)
assert context_hash(contexts) == manifest["sha256_canonical_jsonl"]
assert manifest["seed"] == 42 and manifest["calibration_fraction"] == 0.3
doc_count_distribution = Counter(len(row["docs"]) for row in contexts)
print("Passage-count distribution:", dict(doc_count_distribution))
assert all(sum(row["condition"] == c and row["split"] == "test" for row in contexts) == n for c, n in test_expected.items())
context_keys = {row_key(row) for row in contexts}
assert len(context_keys) == len(contexts)

def check_predictions(rows, name, complete):
    keys = [row_key(row) for row in rows]
    assert not [k for k, n in Counter(keys).items() if n > 1], f"Duplicate {name} rows"
    missing = context_keys - set(keys)
    assert not (set(keys) - context_keys), f"Unexpected {name} rows"
    if complete:
        assert not missing, f"Incomplete {name}: {len(missing)} missing"
    print(f"{name}: {len(rows)} rows; missing={len(missing)}")
    return missing

check_predictions(e0, "E0", complete=True)
check_predictions(e2, "cached E2", complete=True)
assert e0_meta["context_sha256"] == e2_meta["context_sha256"] == manifest["sha256_canonical_jsonl"]
assert e0_meta["seed"] == e2_meta["seed"] == manifest["seed"]
print("Shared context, split, and metadata checks: OK")


Passage-count distribution: {5: 1293, 4: 3, 3: 4}
E0: 1300 rows; missing=0
cached E2: 1300 rows; missing=0
Shared context, split, and metadata checks: OK


In [3]:
contexts_by_key = {row_key(row): row for row in contexts}
score_groups = defaultdict(list)
for row in scores:
    context = contexts_by_key[row_key(row)]
    assert 0 <= row["doc_idx"] < len(context["docs"])
    score_groups[row_key(row)].append(row)
context_doc_counts = {key: len(context["docs"]) for key, context in contexts_by_key.items()}
incomplete_score_groups = [(k, len(v), context_doc_counts[k]) for k, v in score_groups.items() if len(v) != context_doc_counts[k]]
missing_score_groups = [(k, 0) for k in context_keys if k not in score_groups]
score_gaps = incomplete_score_groups + missing_score_groups
if score_gaps:
    print("ERROR: passage-score cache is incomplete:", len(score_gaps), "context groups")
    print("First gaps:", score_gaps[:5])
    raise RuntimeError("Passage-score cache is incomplete; ask Qy for the complete rgb_e2_passage_scores.jsonl before evaluating.")
print("Passage-score alignment: OK; scores match every context's passage count")

filter_threshold = e2_meta["filter_threshold"]
tau = read_json(E2_RESULTS / "rgb_e2_qwen_metrics.json")["abstention_threshold"]
print("Filter threshold:", filter_threshold, " Abstention threshold:", tau)


Passage-score alignment: OK; scores match every context's passage count
Filter threshold: 1.3702995011356746e-07  Abstention threshold: 4.796591794483425e-07


## 1. Cached evaluation

The constants and scoring helpers below are copied verbatim from **E2_Qwen_F1/E2_qwen_f1_filter.ipynb, cell 4**:

- `contains` (answer matching)
- `REFUSAL`, `REFUSAL_KEY`, `REFUSAL_PHRASES`, and `HEDGE_MARKERS`
- `reject` (refusal/hedge detection)
- `summarize`
- `bal`

This evaluation uses cached predictions only. It does not call an LLM or use a GPU.


In [4]:
REFUSAL="I can not answer the question because of insufficient information in documents.";REFUSAL_KEY="insufficient information"
REFUSAL_PHRASES=[REFUSAL_KEY,"can not answer","cannot answer","unable to answer","cannot be answered","cannot be determined","cannot determine","does not provide","do not provide","doesn't provide","does not contain","do not contain","doesn't contain","does not specify","do not specify","doesn't specify","does not mention","do not mention","not mentioned","no information","no mention","none of the provided documents","none of the documents","not explicitly","not directly"]
HEDGE_MARKERS=["likely","probably","presumably","possibly","might","suggest","infer","implied","indicating","as indicated","assume","speculat","expected to","would be","would have","though"]
def contains(pred,ans):
 pred=pred.lower();ans=[ans] if isinstance(ans,str) else ans
 for part in ans:
  alts=part if isinstance(part,list) else [part]
  if not any(str(a).lower() in pred for a in alts):return False
 return True
def reject(pred):
 p=pred.lower()
 if REFUSAL_KEY in p:return True
 if not any(x in p for x in REFUSAL_PHRASES):return False
 return not (any(x in p for x in HEDGE_MARKERS))
def summarize(name,es,pm):
 co=re=wr=0
 for e in es:
  p=pm[(name,str(e["id"]))];ok=contains(p,e["answer"]);r=reject(p) and not ok;co+=ok;re+=r;wr+=not ok and not r
 if name=="incomplete":return {"n":len(es),"rejection_rate":re/max(1,len(es))}
 return {"n":len(es),"accuracy":co/max(1,len(es)),"rejection_rate":re/max(1,len(es)),"wrong_answer":wr/max(1,len(es))}
def bal(s):return (sum(s[n]["accuracy"] for n in ("noise_0.0","noise_0.4","noise_0.8"))/3+s["incomplete"]["rejection_rate"])/2


In [5]:
import csv

metrics = read_json(E2_RESULTS / "rgb_e2_qwen_metrics.json")
tau = metrics["abstention_threshold"]

# Reattach the answer fields used by the reference notebook from its RGB inputs.
def read_json_records(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

refine = read_json_records(REPO_ROOT / "RGB" / "data" / "en_refine.json")
fact = read_json_records(REPO_ROOT / "RGB" / "data" / "en_fact.json")
answers = {}
for condition in ("noise_0.0", "noise_0.4", "noise_0.8", "incomplete"):
    answers.update({(condition, str(row["id"])): row["answer"] for row in refine})
answers.update({("contradictory", str(row["id"])): row["answer"] for row in fact})
eval_rows = []
for row in contexts:
    item = dict(row)
    item["answer"] = answers[row_key(row)]
    eval_rows.append(item)

score_by_key = defaultdict(list)
for row in scores:
    score_by_key[row_key(row)].append(row)
max_original_score = {
    key: max(item["score"] for item in rows)
    for key, rows in score_by_key.items()
}

# This cache is filter + generate only; abstention is applied below.
filter_only = {(row["condition"], str(row["id"])): row["pred"] for row in e2}
e0_predictions = {(row["condition"], str(row["id"])): row["pred"] for row in e0}
e2_filter_only = dict(filter_only)
e2_full = dict(filter_only)
low_score_test = [
    row for row in eval_rows
    if row["split"] == "test" and max_original_score[row_key(row)] < tau
]
refusal_count = sum(e2_filter_only[row_key(row)] == REFUSAL for row in low_score_test)
print(f"Cached E2 exact refusals below tau: {refusal_count}/{len(low_score_test)}")
if low_score_test and refusal_count / len(low_score_test) >= 0.95:
    raise RuntimeError("Nearly all low-score cached E2 answers are already the REFUSAL sentence; stopping because the cache may already include abstention.")
for row in eval_rows:
    key = row_key(row)
    if max_original_score[key] < tau:
        e2_full[key] = REFUSAL

e0_abstain_only = dict(e0_predictions)
e0_abstain_replaced_by_condition = {
    condition: sum(
        max_original_score[row_key(row)] < tau
        for row in eval_rows
        if row["split"] == "test" and row["condition"] == condition
    )
    for condition in ("noise_0.0", "noise_0.4", "noise_0.8", "incomplete", "contradictory")
}
for row in eval_rows:
    key = row_key(row)
    if max_original_score[key] < tau:
        e0_abstain_only[key] = REFUSAL
print(
    "E0 answers replaced by REFUSAL on test split:",
    sum(e0_abstain_replaced_by_condition.values()),
    e0_abstain_replaced_by_condition,
)

conditions = ("noise_0.0", "noise_0.4", "noise_0.8", "incomplete", "contradictory")
test_rows = {condition: [row for row in eval_rows if row["condition"] == condition and row["split"] == "test"] for condition in conditions}
assert {condition: len(rows) for condition, rows in test_rows.items()} == {
    "noise_0.0": 210, "noise_0.4": 210, "noise_0.8": 210, "incomplete": 210, "contradictory": 70
}

summaries = {}
for method, predictions in (("E0", e0_predictions), ("E2_filter_only", e2_filter_only), ("E0_abstain_only", e0_abstain_only), ("E2_full", e2_full)):
    per_condition = {condition: summarize(condition, test_rows[condition], predictions) for condition in conditions}
    summaries[method] = {**per_condition, "balanced": bal(per_condition)}
    print(method, summaries[method])


Cached E2 exact refusals below tau: 113/130
E0 answers replaced by REFUSAL on test split: 130 {'noise_0.0': 1, 'noise_0.4': 2, 'noise_0.8': 18, 'incomplete': 102, 'contradictory': 7}
E0 {'noise_0.0': {'n': 210, 'accuracy': 0.9380952380952381, 'rejection_rate': 0.0380952380952381, 'wrong_answer': 0.023809523809523808}, 'noise_0.4': {'n': 210, 'accuracy': 0.9428571428571428, 'rejection_rate': 0.0380952380952381, 'wrong_answer': 0.01904761904761905}, 'noise_0.8': {'n': 210, 'accuracy': 0.6952380952380952, 'rejection_rate': 0.24285714285714285, 'wrong_answer': 0.06190476190476191}, 'incomplete': {'n': 210, 'rejection_rate': 0.8857142857142857}, 'contradictory': {'n': 70, 'accuracy': 0.5571428571428572, 'rejection_rate': 0.2857142857142857, 'wrong_answer': 0.15714285714285714}, 'balanced': 0.8722222222222222}
E2_filter_only {'noise_0.0': {'n': 210, 'accuracy': 0.9285714285714286, 'rejection_rate': 0.04285714285714286, 'wrong_answer': 0.02857142857142857}, 'noise_0.4': {'n': 210, 'accuracy':

In [6]:
comparison = []
for method, reference_method in (("E0", "E0"), ("E2_full", "E2")):
    reproduced_metrics = summaries[method]
    reference_metrics = metrics["methods"][reference_method]["test"]
    for condition in conditions:
        for metric_name in sorted(set(reproduced_metrics[condition]) | set(reference_metrics[condition])):
            reproduced_value = reproduced_metrics[condition].get(metric_name)
            reference_value = reference_metrics[condition].get(metric_name)
            difference = None if reproduced_value is None or reference_value is None else reproduced_value - reference_value
            comparison.append({
                "method": method, "condition": condition, "metric": metric_name,
                "reproduced_value": reproduced_value, "reference_value": reference_value, "difference": difference,
            })
    comparison.append({
        "method": method, "condition": "all", "metric": "balanced",
        "reproduced_value": reproduced_metrics["balanced"], "reference_value": reference_metrics["balanced"],
        "difference": reproduced_metrics["balanced"] - reference_metrics["balanced"],
    })

comparison_path = E3_RESULTS / "reproduction_check.csv"
with comparison_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["method", "condition", "metric", "reproduced_value", "reference_value", "difference"])
    writer.writeheader()
    writer.writerows(comparison)
print(f"Wrote {comparison_path}")
print("method,condition,metric,reproduced_value,reference_value,difference")
for row in comparison:
    print(",".join(str(row[field]) for field in ("method", "condition", "metric", "reproduced_value", "reference_value", "difference")))

mismatches = [row for row in comparison if row["difference"] is not None and abs(row["difference"]) > 1e-9]
if mismatches:
    print(f"MISMATCHES: {len(mismatches)}")
else:
    print("All E0 and E2_full comparison metrics match the reported test results.")


Wrote /Users/kiki/Desktop/sc4001-reliable-rag/SC4001-Reliable-RAG/results_e3_ablation/reproduction_check.csv
method,condition,metric,reproduced_value,reference_value,difference
E0,noise_0.0,accuracy,0.9380952380952381,0.9380952380952381,0.0
E0,noise_0.0,n,210,210,0
E0,noise_0.0,rejection_rate,0.0380952380952381,0.0380952380952381,0.0
E0,noise_0.0,wrong_answer,0.023809523809523808,0.023809523809523808,0.0
E0,noise_0.4,accuracy,0.9428571428571428,0.9428571428571428,0.0
E0,noise_0.4,n,210,210,0
E0,noise_0.4,rejection_rate,0.0380952380952381,0.0380952380952381,0.0
E0,noise_0.4,wrong_answer,0.01904761904761905,0.01904761904761905,0.0
E0,noise_0.8,accuracy,0.6952380952380952,0.6952380952380952,0.0
E0,noise_0.8,n,210,210,0
E0,noise_0.8,rejection_rate,0.24285714285714285,0.24285714285714285,0.0
E0,noise_0.8,wrong_answer,0.06190476190476191,0.06190476190476191,0.0
E0,incomplete,n,210,210,0
E0,incomplete,rejection_rate,0.8857142857142857,0.8857142857142857,0.0
E0,contradictory,accuracy,0.5571428